# model_dl_001  –  KIER 1D CNN-LSTM / Seq2Seq 예측 모델
`1DCNNLSTM-01/02/03` · `DL-01-01` · `DL-02` · `Comparison-04` 통합

## 01. Init

In [ ]:
#region Basic_Import
## Basic
import os, sys, warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
pd.options.display.float_format = '{:.10f}'.format

## Datetime
from datetime import datetime, date, timedelta

## glob
import glob, requests, json

## 시각화
import matplotlib.pyplot as plt, seaborn as sns
# %matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]

from scipy import stats

## Split, 정규화
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# K-Means 알고리즘
from sklearn.cluster import KMeans, MiniBatchKMeans

# Clustering 알고리즘의 성능 평가 측도
from sklearn import metrics
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score, silhouette_score, rand_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

## For Web

import tqdm
from tqdm.notebook import tqdm
#endregion Basic_Import

In [ ]:
## Import_DL
import tensorflow as tf
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import Sequential, load_model
print('TF version:', tf.__version__)

In [ ]:
## LSTM
from tensorflow import keras
from tensorflow.keras import Sequential, layers, callbacks
from tensorflow.keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional
from keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
## Import_Local
from core import model_dl          as com_DL
from core import model_ml          as com_Model
from core import data_datetime     as com_date
from core import data_analysis     as com_Analysis
from core import provider_kier_m02 as com_KIER_M02
from core import provider_kma      as com_KMA
from core import data_clustering   as com_clustering

In [ ]:
## Init_config
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
## Define Todate str
str_now_ymd = dt.datetime.now().date()
str_now_y, str_now_m, str_now_d = dt.datetime.now().year, dt.datetime.now().month, dt.datetime.now().day
str_now_hr, str_now_min = dt.datetime.now().hour, dt.datetime.now().minute

print(dt.datetime.now())
print(str(str_now_y) + " / " + str(str_now_m)  + " / " + str(str_now_d))
print(str(str_now_hr) + " : " + str(str_now_min))

In [ ]:
## Dict_Domain
int_domain   = 0   # 0=ELEC 1=HEAT 2=WATER 3=HOT_HEAT 4=HOT_FLOW 5=GAS
int_interval = 1   # 0=10min 1=1H 2=1D 3=1W 4=1M
K            = 3   # 군집 수 (clustering_001에서 결정)
int_grp      = 0   # 0=ALL 1=C0 2=C1 3=C2

str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
_, _, _, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
str_file  = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
str_interval_str, *_ = com_KIER_M02.create_file_str(str_domain, int_interval)

## 02. 공통 함수 정의
군집화 로더 / 데이터셋 준비 / Seq2Seq 모델 빌더

In [ ]:
def cluster_label():
    df_kier_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
    df_kier_raw['METER_DATE'] = pd.to_datetime(df_kier_raw['METER_DATE'])

    ## 호실별 순시 사용량 컬럼만 가져오기
    list_col_tar = list(df_kier_raw.columns[1:])
    df_kier_h = df_kier_raw.set_index('METER_DATE')

    # ## Error Log : "[5:-2]" 부분을 추가하여 연월일시 및 평균합계 부분을 제거해주지 않으면, 군집화 계수가 제대로 도출되지 못함.
    # df_kier_summary_total = df_kier_h.transpose().reset_index()[5:-2]
    # ## 또는, 가장 깔끔하게 이렇게 처리해도 좋다
    df_kier_summary_total = df_kier_h[list_col_tar].transpose().reset_index()

    ## 세대 번호의 컬럼명이 'index'로 지정되어 오류 발생
    df_kier_summary_total['h_index'] = df_kier_summary_total['index']
    df_kier_summary_total = df_kier_summary_total.drop(columns = ['index'])

    X = df_kier_summary_total.drop(columns = 'h_index')
    y = df_kier_summary_total['h_index']

    # 변수 표준화
    scaler = StandardScaler() # 변수 표준화 클래스
    scaler.fit(X)  # 표준화를 위해 변수별 파라미터(평균, 표준편차) 계산
    X_std = scaler.transform(X)  # 훈련자료 표준화 변환

    ## 최종 군집에 대한 Labeled Data 저장
    km = KMeans(n_clusters = K, init="k-means++", max_iter=300, n_init=1).fit(X_std)
    list_size_cluster = com_clustering.get_cluster_sizes(km, X_std) ## 최종 군집화에 대한 군집 크기
    df_kier_summary_total['target_'+str_domain] = 0
    for i in range(0, len(df_kier_summary_total)) : df_kier_summary_total['target_'+str_domain].iloc[i] = km.labels_[i]

    str_file_labeled = str_dirName_h + 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
    df_kier_summary_total = df_kier_summary_total[['h_index', 'target_'+str_domain]]
    df_kier_summary_total.to_csv(str_file_labeled)

    return df_kier_summary_total, list_size_cluster

In [ ]:
def load_dataset_Not_cluster():
    ## ▶ Dataset 불러오기
    ## 1. Interpolate / Filled ASOS Data
    str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
    df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

    try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
    except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

    ## 3. 1시간 단위 사용량 Data Load
    str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
    df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)

    ## ▶ h_index에 따라 Dataset 분리
    ## 1. 각 index별 house 목록 생성
    list_kier_h_all = df_kier_h_cluster['h_index']

    ## 2. 전체 사용량 합계 구하기
    df_kier_h_all = df_raw.copy()
    df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_all]
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
    df_kier_h_all.dropna()

    ## 4. 날씨 데이터 추가
    df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
    df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

    # str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]
    str_col_tar = str_domain + '_INST_SUM_ALL'
    df_tar_res = df_kier_h_all.drop(columns = ['METER_DATE', 'DAY']).dropna()

    return df_tar_res, str_col_tar

In [ ]:
def load_dataset_cluster(int_grp):
    ## ▶ Dataset 불러오기
    ## 1. Interpolate / Filled ASOS Data
    str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
    df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

    try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
    except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

    ## 2. Labeled Data Load
    ## Cluster 기준 Interval
    str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
    df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                    , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
    # print(str_interval)
    # print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
    # df_kier_h_cluster

    ## 3. 1시간 단위 사용량 Data Load
    str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
    df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)



    ## ▶ h_index에 따라 Dataset 분리
    ## 1. 각 index별 house 목록 생성
    list_kier_h_all = df_kier_h_cluster['h_index']
    # print(len(list_kier_h_all))
    list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
    # print(len(list_kier_h_c0))
    list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
    # print(len(list_kier_h_c1))

    if K == 3 : list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
    # print(len(list_kier_h_c2))

    ## 2. 전체 사용량 합계 구하기
    df_kier_h_all = df_raw.copy()
    df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_all]
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
    df_kier_h_all.dropna()

    ## 3. Cluster별 사용량 합계 산출
    ## ■ C00
    df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
    df_kier_h_c0['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c0]
    df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_c0[str_domain + '_INST_SUM_C0'].shift(1)
    df_kier_h_c0.dropna()

    ## ■ C01
    df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
    df_kier_h_c1['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c1]
    df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_c1[str_domain + '_INST_SUM_C1'].shift(1)
    df_kier_h_c1.dropna()

    if K == 3:
        ## ■ C02
        df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
        df_kier_h_c2['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
        df_kier_h_tmp = df_raw[list_kier_h_c2]
        df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
        ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
        df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_c2[str_domain + '_INST_SUM_C2'].shift(1)
        df_kier_h_c2.dropna()

    ## 4. 날씨 데이터 추가
    df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
    df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

    df_kier_h_c0 = pd.merge(df_kier_h_c0, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c0 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c0)
    df_kier_h_c0 = com_date.create_col_ymdhm(df_kier_h_c0, 'METER_DATE')

    df_kier_h_c1 = pd.merge(df_kier_h_c1, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c1 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c1)
    df_kier_h_c1 = com_date.create_col_ymdhm(df_kier_h_c1, 'METER_DATE')

    if K == 3:
        df_kier_h_c2 = pd.merge(df_kier_h_c2, df_ASOS, how = 'left', on = ['METER_DATE'])
        df_kier_h_c2 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c2)
        df_kier_h_c2 = com_date.create_col_ymdhm(df_kier_h_c2, 'METER_DATE')

    ## 모든 세대
    if int_grp == 0 : df_tar_res = df_kier_h_all.drop(columns = ['METER_DATE', 'DAY']).dropna()
    ## 군집 C0
    elif int_grp == 1 : df_tar_res = df_kier_h_c0.drop(columns = ['METER_DATE', 'DAY']).dropna()
    ## 군집 C1
    elif int_grp == 2 : df_tar_res = df_kier_h_c1.drop(columns = ['METER_DATE', 'DAY']).dropna()
    ## 군집 C0
    elif int_grp == 3 : df_tar_res = df_kier_h_c2.drop(columns = ['METER_DATE', 'DAY']).dropna()

    str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]

    return df_tar_res, str_col_tar

In [ ]:
def seq2seq_model(input_shape):
    model_input = tf.keras.layers.Input(shape=input_shape)

    # for feature extracting
    conv1 = tf.keras.layers.Conv1D(64, 1, activation='swish')(model_input)
    pool1 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv1)
    bat01 = tf.keras.layers.BatchNormalization()(pool1)
    conv2 = tf.keras.layers.Conv1D(32, 1, activation='swish')(bat01)
    pool2 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv2)
    bat02 = tf.keras.layers.BatchNormalization()(pool2)

    # 인코더 - 디코더 선언
    encoder_lstm1 = tf.keras.layers.LSTM(16, return_sequences=True, activation='swish')
    encoder_lstm2 = tf.keras.layers.LSTM(32, return_sequences=True, activation='swish')
    encoder_lstm3 = tf.keras.layers.LSTM(64, return_state=True, return_sequences=True, activation='swish')

    decoder_lstm1 = tf.keras.layers.LSTM(64, return_sequences=True, activation='swish')
    decoder_lstm2 = tf.keras.layers.LSTM(32, return_sequences=True, activation='swish')
    decoder_lstm3 = tf.keras.layers.LSTM(16, return_sequences=True, activation='swish')

    # 인코더
    encoder_output_lstm1 = encoder_lstm1(bat02)
    encoder_output_lstm2 = encoder_lstm2(bat01)
    encoder_output_lstm4, state_h, state_c = encoder_lstm3(encoder_output_lstm2)

    #디코더
    decoder_lstm1_output = decoder_lstm1(encoder_output_lstm4, initial_state=[state_h, state_c])
    decoder_lstm2_output = decoder_lstm2(decoder_lstm1_output)
    decoder_lstm3_output = decoder_lstm3(decoder_lstm2_output)

    flatten = tf.keras.layers.Flatten()(decoder_lstm3_output)
    model_output = tf.keras.layers.Dense(1)(flatten)
    
    model = tf.keras.models.Model(model_input, model_output)
    
    return model

## 03. 1D CNN-LSTM 단일 실행
인라인 데이터 로드 → 모델 빌드 → 학습 · 평가 · 시각화

In [ ]:
## KMA_ASOS Data
# str_dir_kmaAsos = "../data/data_KMA_ASOS/"

## Interpolate / Filled ASOS Data
str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

df_ASOS

In [ ]:
## Cluster 기준 Interval
str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
print(str_interval)
print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
df_kier_h_cluster

In [ ]:
list_kier_h_all = df_kier_h_cluster['h_index']
print(len(list_kier_h_all))
list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
print(len(list_kier_h_c0))
list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
print(len(list_kier_h_c1))
list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
print(len(list_kier_h_c2))

In [ ]:
## 사용량 Data Load
## 1시간 단위
str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
df_raw

In [ ]:
## 1) Target 추가 : Cluster별 사용량 합계
## 2) 입력변수 추가 : 각 세대별 1시간 이전 사용량
## ■ 전체 사용량 합계
df_kier_h_all = df_raw.copy()
df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_all]
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
df_kier_h_all.dropna()
df_kier_h_all = df_kier_h_all.dropna()

## ■ C00
df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
df_kier_h_c0['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c0]
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_c0[str_domain + '_INST_SUM_C0'].shift(1)
df_kier_h_c0 = df_kier_h_c0.dropna()

## ■ C01
df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
df_kier_h_c1['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c1]
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_c1[str_domain + '_INST_SUM_C1'].shift(1)
df_kier_h_c1 = df_kier_h_c1.dropna()

if K == 3 : 
    ## ■ C02
    df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
    df_kier_h_c2['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c2]
    df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_c2[str_domain + '_INST_SUM_C2'].shift(1)
    df_kier_h_c2 = df_kier_h_c2.dropna()

In [ ]:
df_kier_h_c0

In [ ]:
## 날씨 데이터 추가
df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

df_kier_h_c0 = pd.merge(df_kier_h_c0, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_c0 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c0)
df_kier_h_c0 = com_date.create_col_ymdhm(df_kier_h_c0, 'METER_DATE')

df_kier_h_c1 = pd.merge(df_kier_h_c1, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_c1 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c1)
df_kier_h_c1 = com_date.create_col_ymdhm(df_kier_h_c1, 'METER_DATE')

if K == 3 : 
    df_kier_h_c2 = pd.merge(df_kier_h_c2, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c2 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c2)
    df_kier_h_c2 = com_date.create_col_ymdhm(df_kier_h_c2, 'METER_DATE')

In [ ]:
## 각 세대별 이전시간 사용량 컬럼을 제거하고 분석하는 경우
# df_kier_h_all = df_kier_h_all.drop(labels = list_kier_h_all, axis = 1)
# df_kier_h_c0 = df_kier_h_c0.drop(labels = list_kier_h_c0, axis = 1)
# df_kier_h_c1 = df_kier_h_c1.drop(labels = list_kier_h_c1, axis = 1)
# if K == 3 : df_kier_h_c2 = df_kier_h_c2.drop(labels = list_kier_h_c2, axis = 1)

In [ ]:
# pd.set_option('display.max_row', 500)
print(df_kier_h_all.shape, ' /// ', df_kier_h_all.columns)
print(df_kier_h_all.isna().sum())
print(df_kier_h_c0.shape, ' /// ', df_kier_h_all.columns)
print(df_kier_h_c0.isna().sum())
print(df_kier_h_c1.shape, ' /// ', df_kier_h_all.columns)
print(df_kier_h_c1.isna().sum())

if K == 3 : 
    print(df_kier_h_c2.shape, ' /// ', df_kier_h_all.columns)
    print(df_kier_h_c2.isna().sum())

### 세대 선택 및 모델 실행

In [ ]:
## 모든 세대
# df_raw = df_kier_h_all
# str_col_tar = str_domain + '_INST_SUM_ALL'
## C1 세대
# df_raw = df_kier_h_c0
# str_col_tar = str_domain + '_INST_SUM_C0'
## C2 세대
df_raw = df_kier_h_c1
str_col_tar = str_domain + '_INST_SUM_C1'
## C3 세대
# df_raw = df_kier_h_c2
# str_col_tar = str_domain + '_INST_SUM_C2'

df_raw = df_raw.drop(columns = ['METER_DATE', 'DAY']).dropna()

In [ ]:
seqLength = 3

## 시퀀스 데이터셋 구성 (core/model_dl.py: split_build_dataset)
trainX, testX, trainY, testY = com_DL.split_build_dataset(
    df_raw, 0.3, str_col_tar, seqLength)

In [ ]:
## Train/Test Check
print(trainX.shape, trainY.shape)
print(testX.shape, testY.shape)

int_len_col_input = trainX.shape[2]
int_len_col_input

In [ ]:
## Activation Function List - ['swish', 'relu', 'tanh']
# str_act_func = 'swish'

# model_input = tf.keras.layers.Input(shape=(seqLength, int_len_col_input))

# conv1 = tf.keras.layers.Conv1D(128, 1, activation = str_act_func)(model_input)
# pool1 = tf.keras.layers.MaxPool1D(pool_size = 2, strides = 1)(conv1)
# conv2 = tf.keras.layers.Conv1D(256, 1, activation = str_act_func)(pool1)
# pool2 = tf.keras.layers.MaxPool1D(pool_size = 2, strides = 1)(conv2)
# conv3 = tf.keras.layers.Conv1D(512, 1, activation = str_act_func)(pool2)
# lstm0 = tf.keras.layers.LSTM(512, activation = str_act_func, dropout = 0.15, return_sequences = True)(pool2)
# lstm1 = tf.keras.layers.LSTM(256, activation = str_act_func, dropout = 0.15, return_sequences = True)(lstm0)
# lstm2 = tf.keras.layers.LSTM(128, activation = str_act_func, dropout = 0.15, return_sequences = True)(lstm1)
# dense1 = tf.keras.layers.Dense(128, activation = str_act_func)(lstm2)
# dense2 = tf.keras.layers.Dense(64, activation = str_act_func)(dense1)
# dense3 = tf.keras.layers.Dense(32, activation = str_act_func)(dense2)
# model_output = tf.keras.layers.Dense(1)(dense3)
# model = tf.keras.models.Model(model_input, model_output)

# model.summary()

# earlystopping = EarlyStopping(monitor='loss', patience=15)

# model.compile(loss='mse', optimizer=tf.keras.optimizers.Adamax(learning_rate=1e-5, clipnorm=1.0), metrics=['mae'])
# # model.compile(loss='mae'
# #               , optimizer=tf.keras.optimizers.SGD(learning_rate=1e-5, clipnorm=1.0)
# #               , metrics=['mae'])

# hist = model.fit(trainX, trainY, epochs=100, batch_size=128, callbacks=[earlystopping])

In [ ]:
## 1D-CNN LSTM 모델 (core/model_dl.py: build_1dcnn_lstm)
_, model = com_DL.build_1dcnn_lstm(int_len_col_input, seqLength)
model.summary()

In [ ]:
## 학습 · 예측 (core/model_dl.py: model_dl_predict)
actual, preds, tm_code = com_DL.model_dl_predict(
    trainX, trainY, testX, testY, model)
true = actual
pred = preds

In [ ]:
## 임시조치
## ValueError: Mean Squared Logarithmic Error cannot be used when targets contain negative values.
cnt_negative = 0
for i in range(0, len(pred)) : 
    if pred[i] < 0 : 
        pred[i] = pred[i] * -1
        cnt_negative = cnt_negative + 1

for i in range(0, len(pred)) : 
    if pred[i] < 0 : print(pred[i])

if cnt_negative != 0 : print(cnt_negative)

In [ ]:
com_Model.model_sk_metrics(true, pred)
# list_scores.append(tm_code)

print("time : ", tm_code)

In [ ]:
import matplotlib.font_manager as fm
FONT_TIMES_NEW_ROMAN = fm.FontProperties(fname='../core/Times New Roman.ttf')

str_model = '1DCNN-LSTM'
print(str_domain, ' /// ', str_interval, ' /// ', str_col_tar, ' /// ', str_model)

str_title = str_domain + ' : Energy Usage by ' + str_model + ' (Interval : ' + str_interval + ')'
plt.figure(figsize=(75, 20), dpi = 500)
plt.plot(true, color='red', label='True')
plt.plot(pred, color='blue', label='Pred')
plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.xticks(fontsize=30)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontsize=30)
plt.legend(fontsize = 50)
plt.show()
plt.savefig(str_title + '.png')

str_title = str_domain + ' : Energy Usage on Early Period by ' + str_model + ' (Interval : ' + str_interval + ')'
plt.figure(figsize=(75, 20), dpi = 500)
plt.plot(true, color='red', label='True')
plt.plot(pred, color='blue', label='Pred')
plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.xticks(fontsize=30)
plt.xlim(0, 1000)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontsize=30)
plt.legend(fontsize = 50)
plt.show()
plt.savefig(str_title + '.png')

str_title = str_domain + ' : Energy Usage by ' + str_model + ' (Interval : ' + str_interval + ')'
plt.figure(figsize=(75, 20), dpi = 500)
plt.plot(pred, color='blue', label='Pred')
plt.plot(true, color='red', label='True')
plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.xticks(fontsize=30)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontsize=30)
plt.legend(fontsize = 50)
plt.show()
plt.savefig(str_title + '.png')

str_title = str_domain + ' : Energy Usage on Early Period by ' + str_model + ' (Interval : ' + str_interval + ')'
plt.figure(figsize=(75, 20), dpi = 500)
plt.plot(pred, color='blue', label='Pred')
plt.plot(true, color='red', label='True')
plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.xticks(fontsize=30)
plt.xlim(0, 1000)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontsize=30)
plt.legend(fontsize = 50)
plt.show()
plt.savefig(str_title + '.png')

## 04. 1D CNN-LSTM + 군집화 KFold CV
`DL-01-01` 기준 · cluster_label() 로 동적 군집화

In [ ]:
str_file_asos = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'  ## ▶ ASOS 파일 경로

import sys
from sklearn.model_selection import KFold, TimeSeriesSplit 
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

float_rate = 0.3
int_fold = 10

## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## K : 2 or 3
## {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
## {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
# dict_ml_model = {0 : 'CB', 1 : 'DT', 2 : 'LGBM', 3 : 'RF', 4 : 'XGB'}
dict_dl_model = {0 : '1D-CNN_LSTM', 1 : ''}
dict_interval = {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
dict_grp = {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
int_domain, int_grp = 0, 1

K = 2 ## 2, 3
int_interval = 3 ## 3, 4
int_model = 0 ## 0, 1, 2, 3, 4

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

# print(str(os.listdir(str_dirData)) + "\n")
# print(os.listdir(str_dirName_h))

str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
df_kier_h_cluster

In [ ]:
## ▶ [Step 1] ASOS 데이터 로드
df_ASOS = pd.read_csv(str_file_asos, index_col=0).reset_index()
try:
    df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
except KeyError:
    df_ASOS = com_date.create_col_datetime(
        df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE'
    ).drop(labels=['None'], axis=1)

In [ ]:
## ▶ [Step 2] KIER 1H 순시 사용량 데이터 로드
str_file_1h = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
df_raw = pd.read_csv(str_dirName_h + str_file_1h, index_col=0)

In [ ]:
## ▶ [Step 3] 전체 세대 합계 산출 및 시점 이동 (1-step lag)
list_kier_h_all = df_kier_h_cluster['h_index']
df_kier_h_all = df_raw.copy()
df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_raw[list_kier_h_all].sum(axis=1).shift(1)

In [ ]:
## ▶ [Step 4] 날씨 데이터 추가 (Merge → Interpolation → 날짜 컬럼 생성)
df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how='left', on=['METER_DATE'])
df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

In [ ]:
## ▶ [Step 5] Target 컬럼 설정 및 최종 Dataset 구성
str_col_tar = str_domain + '_INST_SUM_ALL'
df_tar = df_kier_h_all.drop(columns=['METER_DATE', 'DAY']).dropna()
df_tar.shape

In [ ]:
seqLength = 3

In [ ]:
## Data Split
trainSet, testSet = train_test_split(df_tar, test_size = float_rate, shuffle = False)

## Input / Target Split
trainXX, trainYY = trainSet.drop([str_col_tar], axis=1), trainSet[[str_col_tar]]
testXX, testYY  = testSet.drop([str_col_tar],  axis=1), testSet[[str_col_tar]]

In [ ]:
trainXXcolumns = trainXX.columns
int_len_col_input = len(trainXXcolumns)

In [ ]:
## 비군집화 데이터셋 KFold 실행
sys.stdout.flush()

## 1D-CNN LSTM 모델 빌더 (core/model_dl.py: build_1dcnn_lstm)
n_features_all = len(df_tar.drop(columns=[str_col_tar]).columns)
model_builder = lambda: com_DL.build_1dcnn_lstm(n_features_all, seqLength)[1]
str_model = '1D-CNN_LSTM'

## KFold 분석 (core/model_dl.py: model_dl_analysis_with_KFold)
list_res, list_hists = com_DL.model_dl_analysis_with_KFold(
    df_tar, model_builder, float_rate, str_col_tar, int_fold, seqLength, shuffle=True)

## list_res 저장
str_txt = '../kf_result_include_Clustering_' + str_model + '/kf_result_' + str(dict_interval[int_interval]) + '_ALL_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt'
file_txt = open(str_txt, 'w')
print('- Interval = ' + dict_interval[int_interval] + '\n'
        + '- K = 0' + '\n'
        + '- grp = ALL' + '\n'
        + '- model = ' + str_model + '\n'
        + '- Case = ALL' + ',' + ' size_cluster = ' + str(348) + '\n'
        + '- Size = ' + str(df_tar.shape) + '\n'
        + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
print(list_res, file = file_txt)

## list_hist 저장
str_txt = '../kf_hist_include_Clustering_' + str_model + '/kf_hist_' + str(dict_interval[int_interval]) + '_ALL_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt'
file_txt = open(str_txt, 'w')
print('- Interval = ' + dict_interval[int_interval] + '\n'
        + '- K = 0' + '\n'
        + '- grp = ALL' + '\n'
        + '- model = ' + str_model + '\n'
        + '- Case = ALL' + ',' + ' size_cluster = ' + str(348) + '\n'
        + '- Size = ' + str(df_tar.shape) + '\n'
        + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
print(list_hists, file = file_txt)

file_txt.close()

In [ ]:
## 군집화 데이터셋에 대한 별도 처리
for i in range(0, 3):  ## 각 기간별 3번의 Clustering 병행
    sys.stdout.flush()
    df_kier_summary_total, list_size_cluster = cluster_label()
    print(list_size_cluster)

    ## ▶ [Step 6] Cluster별 house 목록 분리
    list_kier_h_c0 = df_kier_summary_total[df_kier_summary_total['target_' + str_domain] == 0]['h_index']
    list_kier_h_c1 = df_kier_summary_total[df_kier_summary_total['target_' + str_domain] == 1]['h_index']
    if K == 3:
        list_kier_h_c2 = df_kier_summary_total[df_kier_summary_total['target_' + str_domain] == 2]['h_index']

    ## ▶ [Step 7-9] Cluster별 합계 산출, 날씨 추가, DataFrame 매핑
    list_h_c = [list_kier_h_c0, list_kier_h_c1] + ([list_kier_h_c2] if K == 3 else [])
    list_sfx  = ['C0',           'C1']           + (['C2']           if K == 3 else [])
    dict_grp_df = {}
    for c_idx, (list_h, c_sfx) in enumerate(zip(list_h_c, list_sfx), start=1):
        df_c = df_raw.copy()[list_h]
        df_c['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
        df_c[str_domain + '_INST_SUM_' + c_sfx] = df_raw[list_h].sum(axis=1).shift(1)
        df_c = pd.merge(df_c, df_ASOS, how='left', on=['METER_DATE'])
        df_c = com_KMA.Interpolate_KMA_ASOS(df_c)
        df_c = com_date.create_col_ymdhm(df_c, 'METER_DATE')
        dict_grp_df[c_idx] = df_c

    for int_grp in range(1, K + 1):
        print('■ ' + str(int_grp))

        df_tar = dict_grp_df[int_grp].drop(columns=['METER_DATE', 'DAY']).dropna()
        str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]

        ## 1D-CNN LSTM 모델 빌더 (core/model_dl.py: build_1dcnn_lstm)
        int_len_col_input = len(df_tar.drop(columns=[str_col_tar]).columns)
        str_model = '1D-CNN_LSTM'
        model_builder = lambda: com_DL.build_1dcnn_lstm(int_len_col_input, seqLength)[1]

        ## KFold 분석 (core/model_dl.py: model_dl_analysis_with_KFold)
        list_res, list_hists = com_DL.model_dl_analysis_with_KFold(
            df_tar, model_builder, float_rate, str_col_tar, int_fold, seqLength, shuffle=True)

        ## list_res 저장
        str_txt = ('../kf_result_include_Clustering_' + str_model + '/kf_result_'
                   + str(dict_interval[int_interval]) + '_K' + str(K)
                   + '_Case0' + str(i) + '_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt')
        file_txt = open(str_txt, 'w')
        print('- Interval = ' + dict_interval[int_interval] + '\n'
              + '- K = ' + str(K) + '\n'
              + '- grp = C0' + str(int_grp) + '\n'
              + '- model = ' + str_model + '\n'
              + '- Case = 0' + str(i) + ',' + ' size_cluster = ' + str(list_size_cluster) + '\n'
              + '- Size = ' + str(df_tar.shape) + '\n'
              + '- Columns = ' + str(df_tar.columns) + '\n', file=file_txt)
        print(list_res, file=file_txt)

        ## list_hist 저장
        str_txt = ('../kf_hist_include_Clustering_' + str_model + '/kf_hist_'
                   + str(dict_interval[int_interval]) + '_K' + str(K)
                   + '_Case0' + str(i) + '_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt')
        file_txt = open(str_txt, 'w')
        print('- Interval = ' + dict_interval[int_interval] + '\n'
              + '- K = ' + str(K) + '\n'
              + '- grp = C0' + str(int_grp) + '\n'
              + '- model = ' + str_model + '\n'
              + '- Case = 0' + str(i) + ',' + ' size_cluster = ' + str(list_size_cluster) + '\n'
              + '- Size = ' + str(df_tar.shape) + '\n'
              + '- Columns = ' + str(df_tar.columns) + '\n', file=file_txt)
        print(list_hists, file=file_txt)

        file_txt.close()

## 05. 1D CNN-Seq2Seq + 군집화 CV
`DL-02` 기준 · load_dataset_* 함수 사용

In [ ]:
import sys
from sklearn.model_selection import KFold, TimeSeriesSplit 
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

float_rate = 0.3
# test_size = round(len(df_tar) * float_rate)
int_fold = 10

## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## K : 2 or 3
## {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
## {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
# dict_ml_model = {0 : 'CB', 1 : 'DT', 2 : 'LGBM', 3 : 'RF', 4 : 'XGB'}
dict_dl_model = {0 : '1D-CNN_LSTM', 1 : ''}
dict_interval = {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
dict_grp = {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
int_domain, int_grp = 0, 1

K = 3 ## 2, 3
int_interval = 3 ## 3, 4
int_model = 0 ## 0, 1, 2, 3, 4

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

# print(str(os.listdir(str_dirData)) + "\n")
# print(os.listdir(str_dirName_h))

str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
df_kier_h_cluster

In [ ]:
## 비군집화 데이터셋에 대한 별도 처리 (비교군)
sys.stdout.flush() ## flush
df_tar, str_col_tar = load_dataset_Not_cluster()
seqLength = 24

In [ ]:
## 데이터셋 분할 및 시퀀스 구성 (core/model_dl.py: split_build_dataset)
trainX, testX, trainY, testY = com_DL.split_build_dataset(
    df_tar, float_rate, str_col_tar, seqLength)

trainXXcolumns = df_tar.drop(columns=[str_col_tar]).columns
int_len_col_input = trainX.shape[2]

## Train/Test Check
print(trainX.shape, trainY.shape)
print(testX.shape, testY.shape)

In [ ]:
## 1D-CNN Seq2Seq 모델 (core/model_dl.py: build_1dcnn_seq2seq)
str_model, model = com_DL.build_1dcnn_seq2seq(input_shape=(seqLength, int_len_col_input))
# d_actual, model_preds, tm_code = com_DL.model_dl_predict(trainX, trainY, testX, testY, model)

In [ ]:
# ## 임시조치
# ## ValueError: Mean Squared Logarithmic Error cannot be used when targets contain negative values.
# cnt_negative = 0
# for i in range(0, len(model_preds)) : 
#     if model_preds[i] < 0 : 
#         model_preds[i] = model_preds[i] * -1
#         cnt_negative = cnt_negative + 1

# for i in range(0, len(model_preds)) : 
#     if model_preds[i] < 0 : print(model_preds[i])

# if cnt_negative != 0 : print(cnt_negative)

In [ ]:
# list_scores = com_DL.model_sk_metrics(d_actual, model_preds)
# list_scores.append(tm_code)

# print(list_scores)

In [ ]:
## 비군집화 데이터셋에 대한 별도 처리 (비교군)
sys.stdout.flush()
df_tar, str_col_tar = load_dataset_Not_cluster()

## 1D-CNN Seq2Seq 모델 빌더 (core/model_dl.py: build_1dcnn_seq2seq)
n_features_all = len(df_tar.drop(columns=[str_col_tar]).columns)
model_builder = lambda: com_DL.build_1dcnn_seq2seq((seqLength, n_features_all))[1]
str_model = '1D-CNN_Seq2Seq'

## KFold 분석 (core/model_dl.py: model_dl_analysis_with_KFold)
list_res, list_hists = com_DL.model_dl_analysis_with_KFold(
    df_tar, model_builder, float_rate, str_col_tar, int_fold, seqLength, shuffle=True)

## list_res 저장
str_txt = '../kf_result_include_Clustering_' + str_model + '/kf_result_' + str(dict_interval[int_interval]) + '_ALL_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt'
file_txt = open(str_txt, 'w')
print('- Interval = ' + dict_interval[int_interval] + '\n'
        + '- K = 0' + '\n'
        + '- grp = ALL' + '\n'
        + '- model = ' + str_model + '\n'
        + '- Case = ALL' + ',' + ' size_cluster = ' + str(348) + '\n'
        + '- Size = ' + str(df_tar.shape) + '\n'
        + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
print(list_res, file = file_txt)

## list_hist 저장
str_txt = '../kf_hist_include_Clustering_' + str_model + '/kf_hist_' + str(dict_interval[int_interval]) + '_ALL_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt'
file_txt = open(str_txt, 'w')
print('- Interval = ' + dict_interval[int_interval] + '\n'
        + '- K = 0' + '\n'
        + '- grp = ALL' + '\n'
        + '- model = ' + str_model + '\n'
        + '- Case = ALL' + ',' + ' size_cluster = ' + str(348) + '\n'
        + '- Size = ' + str(df_tar.shape) + '\n'
        + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
print(list_hists, file = file_txt)

file_txt.close()

In [ ]:
## 군집화 데이터셋에 대한 별도 처리
for i in range(0, 3):  ## 각 기간별 N번의 Clustering을 병행
    sys.stdout.flush()
    df_kier_summary_total, list_size_cluster = cluster_label()
    print(list_size_cluster)

    for int_grp in range(1, K + 1):  ## 군집 형성된 데이터셋만 분석
        print('■ ' + str(int_grp))
        df_tar, str_col_tar = load_dataset_cluster(int_grp)

        ## 시퀀스 구성 (core/model_dl.py: split_build_dataset)
        trainX, testX, trainY, testY = com_DL.split_build_dataset(
            df_tar, float_rate, str_col_tar, seqLength)
        int_len_col_input = trainX.shape[2]

        ## 1D-CNN Seq2Seq 모델 빌더 (core/model_dl.py: build_1dcnn_seq2seq)
        str_model = '1D-CNN_Seq2Seq'
        model_builder = lambda: com_DL.build_1dcnn_seq2seq((seqLength, int_len_col_input))[1]

        ## KFold 분석 (core/model_dl.py: model_dl_analysis_with_KFold)
        list_res, list_hists = com_DL.model_dl_analysis_with_KFold(
            df_tar, model_builder, float_rate, str_col_tar, int_fold, seqLength, shuffle=True)

        ## list_res 저장
        str_txt = ('../kf_result_include_Clustering_' + str_model + '/kf_result_'
                   + str(dict_interval[int_interval]) + '_K' + str(K)
                   + '_Case0' + str(i) + '_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt')
        file_txt = open(str_txt, 'w')
        print('- Interval = ' + dict_interval[int_interval] + '\n'
              + '- K = ' + str(K) + '\n'
              + '- grp = C0' + str(int_grp) + '\n'
              + '- model = ' + str_model + '\n'
              + '- Case = 0' + str(i) + ',' + ' size_cluster = ' + str(list_size_cluster) + '\n'
              + '- Size = ' + str(df_tar.shape) + '\n'
              + '- Columns = ' + str(df_tar.columns) + '\n', file=file_txt)
        print(list_res, file=file_txt)

        ## list_hist 저장
        str_txt = ('../kf_hist_include_Clustering_' + str_model + '/kf_hist_'
                   + str(dict_interval[int_interval]) + '_K' + str(K)
                   + '_Case0' + str(i) + '_' + dict_grp[int_grp] + '_' + str_model + '_CV' + str(int_fold) + '.txt')
        file_txt = open(str_txt, 'w')
        print('- Interval = ' + dict_interval[int_interval] + '\n'
              + '- K = ' + str(K) + '\n'
              + '- grp = C0' + str(int_grp) + '\n'
              + '- model = ' + str_model + '\n'
              + '- Case = 0' + str(i) + ',' + ' size_cluster = ' + str(list_size_cluster) + '\n'
              + '- Size = ' + str(df_tar.shape) + '\n'
              + '- Columns = ' + str(df_tar.columns) + '\n', file=file_txt)
        print(list_hists, file=file_txt)

        file_txt.close()

## 06. 결과 비교
군집화 유무 · 군집 그룹별 예측 성능 비교 (`1DCNNLSTM-04` 기준)

In [ ]:
import time

In [ ]:
def load_dataset_cluster(int_grp):
    ## ▶ Dataset 불러오기
    if int_grp == 0 : df_raw, str_col_tar = df_kier_h_all, str_domain + '_INST_SUM_ALL'
    elif int_grp == 1 : df_raw, str_col_tar = df_kier_h_c0, str_domain + '_INST_SUM_C0'
    elif int_grp == 2 : df_raw, str_col_tar = df_kier_h_c1, str_domain + '_INST_SUM_C1'
    elif int_grp == 3 : df_raw, str_col_tar = df_kier_h_c2, str_domain + '_INST_SUM_C2'

    df_raw = df_raw.drop(columns = ['METER_DATE', 'DAY']).dropna()
    seqLength = 3

    ## ▶ Data Split + Sequence Building
    trainX, testX, trainY, testY = com_DL.split_build_dataset(df_raw, 0.3, str_col_tar, seqLength)
    int_len_col_input = trainX.shape[2]

    ## ▶ Modeling
    _, model = com_DL.build_1dcnn_lstm(int_len_col_input, seqLength)

    ## ▶ Train & Predict
    true, pred, tm_code = com_DL.model_dl_predict(trainX, trainY, testX, testY, model)

    ## 임시조치
    ## ValueError: Mean Squared Logarithmic Error cannot be used when targets contain negative values.
    cnt_negative = 0
    for i in range(0, len(pred)) : 
        if pred[i] < 0 : 
            pred[i] = pred[i] * -1
            cnt_negative = cnt_negative + 1

    for i in range(0, len(pred)) : 
        if pred[i] < 0 : print(pred[i])

    if cnt_negative != 0 : print(cnt_negative)

    return true, pred, tm_code

In [ ]:
## 비군집화 데이터셋에 대한 처리 (비교군)
sys.stdout.flush() ## flush
d_actual, model_preds, tm_code = load_dataset_cluster(0)

## 군집화 데이터셋에 대한 처리 (실험군)
df_kier_summary_total, list_size_cluster = cluster_label()
print(list_size_cluster)
# df_kier_summary_total

In [ ]:
int_grp = 0
for int_grp in range(1, K + 1): ## 군집 형성된 데이터셋만 분석
    # print('■ ' + str(int_grp))
    # df_tar, str_col_tar = load_dataset_cluster(int_grp) ## 해당 군집에 대한 데이터셋 및 Target Column
    # print(df_tar.columns)
    # print(df_tar.shape)

    if int_grp == 1 : d_actual_clust_01, model_preds_clust_01, tm_code_clust_01 = load_dataset_cluster(1)
    elif int_grp == 2 : d_actual_clust_02, model_preds_clust_02, tm_code_clust_02 = load_dataset_cluster(2)
    elif int_grp == 3 : d_actual_clust_03, model_preds_clust_03, tm_code_clust_03 = load_dataset_cluster(3)
print('D_Completed')
model_preds_clust = []
for i in range(0, len(model_preds_clust_01)):
        if K == 2 : model_preds_clust.append(model_preds_clust_01[i] + model_preds_clust_02[i])
        elif K == 3 : model_preds_clust.append(model_preds_clust_01[i] + model_preds_clust_02[i] + model_preds_clust_03[i])
print(list(model_preds_clust))

In [ ]:
## 시각화 파라미터
## dpi
int_dpi = 300
## Legend 관련
if K == 2 : str_ncol, tuple_axis_allcase = 6, (0.05, -0.2)
elif K == 3 : str_ncol, tuple_axis_allcase = 4, (0.20, -0.3)
tuple_axis_2case = (0.375, -0.2)
tuple_axis_3case = (0.275, -0.2)

In [ ]:
## 대조군 : 군집화 X
## 실험군 01-01 : C0 군집의 합
## 실험군 01-02 : C1 군집의 합
## 실험군 01-03 : C2 군집의 합
## 실험군 02 : 모든 군집의 합
list_scores = com_ML.model_sk_metrics(d_actual, model_preds)
list_scores_c0 = com_ML.model_sk_metrics(d_actual_clust_01, model_preds_clust_01)
list_scores_c1 = com_ML.model_sk_metrics(d_actual_clust_02, model_preds_clust_02)
if K == 3 : list_scores_c2 = com_ML.model_sk_metrics(d_actual_clust_03, model_preds_clust_03)
list_scores_clustered = com_ML.model_sk_metrics(d_actual[:-1], model_preds_clust)

str_txt = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval[1:] + '.txt'
file_txt = open(str_txt, 'w')

print("■ 군집크기", file = file_txt)
print(list_size_cluster, file = file_txt)

print("■ 대조군 : 군집화 X", file = file_txt)
print(list_scores, file = file_txt)

print("■ 실험군 01-01 : C0 군집의 합", file = file_txt)
print(list_scores_c0, file = file_txt)
print("■ 실험군 01-02 : C1 군집의 합", file = file_txt)
print(list_scores_c1, file = file_txt)
if K == 3 : 
    print("■ 실험군 01-03 : C2 군집의 합", file = file_txt)
    print(list_scores_c2, file = file_txt)

print("■ 실험군 02 : 모든 군집의 합", file = file_txt)
print(list_scores_clustered, file = file_txt)

file_txt.close()

In [ ]:
## 1차 비교 (각 실험군 02(모든 군집의 합)과 대조군(군집화 X) 예측 결과와 비교)
str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - All of Group)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

## 대조군
plt.plot(d_actual, color = 'black', linewidth = '7', label='Actual')
plt.plot(model_preds, color = 'red', linewidth = '12', label='Pred', alpha=0.7)
## C0
plt.plot(d_actual_clust_01, color = 'black', linewidth = '7', label='Actual_C0')
plt.plot(model_preds_clust_01, color = 'limegreen', linewidth = '12', label='Pred_C0', alpha=0.5)
## C1
plt.plot(d_actual_clust_02, color = 'black', linewidth = '7', label='Actual_C1')
plt.plot(model_preds_clust_02, color = '#e35f62', linewidth = '12', label='Pred_C1', alpha=0.5)
## C2
if K == 3 : 
    plt.plot(d_actual_clust_03, color = 'black', linewidth = '7', label='Actual_C2')
    plt.plot(model_preds_clust_03, color = 'blue', linewidth = '12', label='Pred_C2', alpha=0.5)

plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontsize = 50)
plt.yticks(fontsize = 50)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = tuple_axis_allcase, ncol = str_ncol, fontsize = 50)
str_plt_title = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval[1:] + '_0-0_ALL Pred.png'
# plt.savefig(str_plt_title, dpi = 500)
print(str_plt_title)

In [ ]:
## 1-1 비교 (각 실험군 02(모든 군집의 합) 비교)
str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (General Method)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

## 대조군
plt.plot(d_actual, color = 'black', linewidth = '7', label='Actual')
plt.plot(model_preds, color = 'red', linewidth = '12', label='Pred', alpha=0.5)

plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontsize = 50)
plt.yticks(fontsize = 50)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = (0.375, -0.2), ncol = str_ncol, fontsize = 50)
str_plt_title = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval[1:] + '_1-0_General Pred.png'
# plt.savefig(str_plt_title, dpi = 700)
print(str_plt_title)

In [ ]:
## C0
str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Group 01)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

plt.plot(d_actual_clust_01, color = 'black', linewidth = '5', label='Actual_C0')
plt.plot(model_preds_clust_01, color = 'limegreen', linewidth = '5', label='Pred_C0', alpha=0.5)
plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontsize = 50)
plt.yticks(fontsize = 50)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = (0.325, -0.2), ncol = str_ncol, fontsize = 50)
str_plt_title = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval[1:] + '_1-1_C0 Pred.png'
# plt.savefig(str_plt_title, dpi = 700)
print(str_plt_title)



str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Group 02)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

## C1
plt.plot(d_actual_clust_02, color = 'black', linewidth = '5', label='Actual_C1')
plt.plot(model_preds_clust_02, color = '#e35f62', linewidth = '5', label='Pred_C1', alpha=0.5)
plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontsize = 50)
plt.yticks(fontsize = 50)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = (0.325, -0.2), ncol = str_ncol, fontsize = 50)
str_plt_title = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval[1:] + '_1-2_C1 Pred.png'
# plt.savefig(str_plt_title, dpi = 700)
print(str_plt_title)



## C2
if K == 3 : 
    str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Group 03)' + "\n"
    plt.figure(figsize=(50, 22.5), dpi = int_dpi)
    print(str_name_figure)

    plt.plot(model_preds_clust_03, color = 'blue', linewidth = '5', label='Pred_C2', alpha=0.5)
    plt.plot(d_actual_clust_03, color = 'black', linewidth = '5', label='Actual_C2')
    plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
    plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
    plt.xticks(fontsize = 50)
    plt.yticks(fontsize = 50)
    plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
    plt.xlim(0, 500)
    plt.legend(loc = (0.325, -0.2), ncol = str_ncol, fontsize = 50)
    str_plt_title = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval[1:] + '_1-3_C2 Pred.png'
    # plt.savefig(str_plt_title, dpi = 700)
    print(str_plt_title)

In [ ]:
## 2차 비교 (실험군 02(모든 군집의 합)과 대조군(군집화 X) 예측 결과와 비교)
str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Sum of Clustered Prediction)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

plt.plot(model_preds, color = '#e35f62', linewidth = '5', label='Pred')
plt.plot(model_preds_clust, color = 'limegreen', linewidth = '5', label='Pred_Clustered')
plt.plot(d_actual, color = 'black', linewidth = '5', label='Actual')

plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontsize = 50)
plt.yticks(fontsize = 50)
plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = tuple_axis_3case, ncol = str_ncol, fontsize = 50)
str_plt_title = 'ELEC_Comparison_1DCNN-LSTM_K' + str(K) + str_interval + '_2-1_Sum Pred.png'
# plt.savefig(str_plt_title, dpi = 700)
print(str_plt_title)

In [ ]:
# import matplotlib.font_manager as fm
# FONT_TIMES_NEW_ROMAN = fm.FontProperties(fname='../core/Times New Roman.ttf')

# str_model = '1DCNN-LSTM'
# print(str_domain, ' /// ', str_interval, ' /// ', str_col_tar, ' /// ', str_model)

# str_title = str_domain + '_Comparison_' + str_model + '_K' + str(K) + str(str_interval) + '_0-0_ALL Pred'
# str_title = str_domain + '_Comparison_' + str_model + '_K' + str(K) + str(str_interval) + '_1-0_General Pred'
# str_title = str_domain + '_Comparison_' + str_model + '_K' + str(K) + str(str_interval) + '_1-1_C0 Pred'
# str_title = str_domain + '_Comparison_' + str_model + '_K' + str(K) + str(str_interval) + '_1-2_C1 Pred'
# str_title = str_domain + '_Comparison_' + str_model + '_K' + str(K) + str(str_interval) + '_1-3_C2 Pred'
# str_title = str_domain + '_Comparison_' + str_model + '_K' + str(K) + str(str_interval) + '_2-1_Sum Pred'
# plt.figure(figsize=(75, 20), dpi = 500)
# plt.plot(true, color='red', label='True')
# plt.plot(pred, color='blue', label='Pred')
# plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
# plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.xticks(fontsize=30)
# plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.yticks(fontsize=30)
# plt.legend(fontsize = 50)
# plt.show()
# plt.savefig(str_title + '.png')

# str_title = str_domain + ' : Energy Usage on Early Period by ' + str_model + ' (Interval : ' + str_interval + ')'
# plt.figure(figsize=(75, 20), dpi = 500)
# plt.plot(true, color='red', label='True')
# plt.plot(pred, color='blue', label='Pred')
# plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
# plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.xticks(fontsize=30)
# plt.xlim(0, 1000)
# plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.yticks(fontsize=30)
# plt.legend(fontsize = 50)
# plt.show()
# plt.savefig(str_title + '.png')

# # str_title = str_domain + ' : Energy Usage by ' + str_model + ' (Interval : ' + str_interval + ')'
# # plt.figure(figsize=(75, 20), dpi = 500)
# # plt.plot(pred, color='blue', label='Pred')
# # plt.plot(true, color='red', label='True')
# # plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
# # plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# # plt.xticks(fontsize=30)
# # plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# # plt.yticks(fontsize=30)
# # plt.legend(fontsize = 50)
# # plt.show()
# # plt.savefig(str_title + '.png')

# # str_title = str_domain + ' : Energy Usage on Early Period by ' + str_model + ' (Interval : ' + str_interval + ')'
# # plt.figure(figsize=(75, 20), dpi = 500)
# # plt.plot(pred, color='blue', label='Pred')
# # plt.plot(true, color='red', label='True')
# # plt.title(str_title, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 80)
# # plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# # plt.xticks(fontsize=30)
# # plt.xlim(0, 1000)
# # plt.ylabel('Energy Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# # plt.yticks(fontsize=30)
# # plt.legend(fontsize = 50)
# # plt.show()
# # plt.savefig(str_title + '.png')